# LeetCode 4110 — Find the Maximum Length of Valid Subsequence III (Medium)

Bài toán LeetCode:

```python
class Solution:
    def maximumLength(self, nums: List[int], k: int) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4110 (trên LeetCode bài này thuộc nhóm Premium).

Bài này kết hợp hai ý tưởng rất hay gặp:

- **Số học đồng dư (modulo):** "tổng chia hết cho `k`" chỉ phụ thuộc vào **số dư** của từng số.
- **Quy hoạch động "kết thúc tại"** kiểu dãy con tăng dài nhất (LIS), rồi tối ưu bằng cách **gom trạng thái theo số dư**.

Cuối cùng ta còn có một lời giải tham lam rất đẹp: **đếm số "khối" xen kẽ**.

---

## Sau notebook này, bạn sẽ nắm được

1. Phân biệt **subsequence** (dãy con, được bỏ phần tử) với **subarray** (đoạn con liên tiếp).
2. Tính chất `(a + b) % k == ((a % k) + (b % k)) % k` và cách tìm **số dư bù** `(k - r) % k`.
3. Brute Force `O(2^n · n)` và DP `O(n²)` kiểu LIS.
4. DP **theo số dư** `O(n)`: `best[r]` = độ dài dãy hợp lệ dài nhất kết thúc bằng một số có số dư `r`.
5. Lời giải tham lam: dãy hợp lệ **xen kẽ hai lớp số dư** → đếm số khối.
6. Lỗi hay gặp (quên `% k` khi tính số dư bù, nhầm subarray...), mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho mảng số nguyên `nums` và số nguyên dương `k`.

Một **subsequence** (dãy con) của `nums` được gọi là **hợp lệ** nếu **tổng của mọi cặp phần tử kề nhau** trong dãy con đều **chia hết cho `k`**. Dãy con chỉ có **một phần tử** luôn hợp lệ (không có cặp kề nhau nào).

Trả về **độ dài lớn nhất** của một dãy con hợp lệ.

---

## Ví dụ 1

```python
nums = [1, 2, 3, 4, 5], k = 3   ->   4
```

Chọn `[1, 2, 4, 5]`: các tổng kề nhau là `1+2 = 3`, `2+4 = 6`, `4+5 = 9` — đều chia hết cho `3`. Không có dãy con hợp lệ nào dài 5 (vì `3` không ghép được với ai bên cạnh).

## Ví dụ 2

```python
nums = [5, 10, 15], k = 5   ->   3
```

Lấy cả mảng: `5+10 = 15`, `10+15 = 25` đều chia hết cho `5`. (Mọi phần tử đều là bội của `5`.)

## Ví dụ 3

```python
nums = [1, 3, 5, 7, 9], k = 4   ->   5
```

Lấy cả mảng: `1+3 = 4`, `3+5 = 8`, `5+7 = 12`, `7+9 = 16` đều chia hết cho `4`. Số dư khi chia cho `4` xen kẽ `1, 3, 1, 3, 1`.

---

## Ràng buộc

```text
2 <= nums.length <= 10^5
1 <= nums[i] <= 10^9
1 <= k <= 10^5
```

# 2. Kiến thức nền tảng

## 2.1. Subsequence khác subarray

| Khái niệm | Định nghĩa | Ví dụ với `[1, 2, 3, 4, 5]` |
|---|---|---|
| **subarray** (đoạn con) | các phần tử **liên tiếp** | `[2, 3, 4]` |
| **subsequence** (dãy con) | giữ **thứ tự**, được **bỏ** phần tử ở giữa | `[1, 2, 4, 5]` (bỏ `3`) |

Bài này là **subsequence**: ở ví dụ 1, ta bỏ số `3` để `2` và `4` trở thành kề nhau.

## 2.2. Phép chia lấy dư `%` và tổng chia hết

`a % k` là số dư khi chia `a` cho `k`, luôn nằm trong `0..k-1` (với `k > 0`).

Tính chất quan trọng nhất:

```text
(a + b) % k == ((a % k) + (b % k)) % k
```

Nghĩa là: muốn biết `a + b` có chia hết cho `k` không, **chỉ cần biết số dư của `a` và `b`**. Giá trị thật (có thể tới `10^9`) không quan trọng.

In [1]:
k = 3
for a, b in [(1, 2), (2, 4), (4, 5), (2, 3), (10, 8)]:
    print(f"a = {a:>2}, b = {b}: a % k = {a % k}, b % k = {b % k} | "
          f"(a + b) % k = {(a + b) % k} | ((a%k) + (b%k)) % k = {((a % k) + (b % k)) % k}")

a =  1, b = 2: a % k = 1, b % k = 2 | (a + b) % k = 0 | ((a%k) + (b%k)) % k = 0
a =  2, b = 4: a % k = 2, b % k = 1 | (a + b) % k = 0 | ((a%k) + (b%k)) % k = 0
a =  4, b = 5: a % k = 1, b % k = 2 | (a + b) % k = 0 | ((a%k) + (b%k)) % k = 0
a =  2, b = 3: a % k = 2, b % k = 0 | (a + b) % k = 2 | ((a%k) + (b%k)) % k = 2
a = 10, b = 8: a % k = 1, b % k = 2 | (a + b) % k = 0 | ((a%k) + (b%k)) % k = 0


## 2.3. Số dư bù: số dư nào ghép được với `r`?

Gọi `r = a % k`. Ta cần `b` sao cho `(r + b % k) % k == 0`, tức là `b % k` phải bằng:

```text
need = (k - r) % k
```

Vì sao phải có `% k` ở cuối? Khi `r = 0`, `k - r = k` — nhưng số dư không bao giờ bằng `k` (chỉ từ `0` tới `k - 1`). Số ghép được với một bội của `k` phải **cũng là bội của `k`**, tức là số dư `0`. `(k - 0) % k = 0` cho đúng kết quả.

In [2]:
k = 4
for r in range(k):
    print(f"r = {r}: k - r = {k - r}, need = (k - r) % k = {(k - r) % k}")

r = 0: k - r = 4, need = (k - r) % k = 0
r = 1: k - r = 3, need = (k - r) % k = 3
r = 2: k - r = 2, need = (k - r) % k = 2
r = 3: k - r = 1, need = (k - r) % k = 1


Để ý quan hệ **đối xứng**: nếu `need(x) = y` thì `need(y) = x`. Với `k = 4`: `1 ↔ 3`, còn `0` và `2` tự ghép với chính nó (`0 + 0 = 0`, `2 + 2 = 4`).

## 2.4. `dict.get(key, default)`

`d.get(key, 0)` trả về `d[key]` nếu có, ngược lại trả về `0` — không báo lỗi `KeyError`. Rất tiện khi dùng dict làm bảng DP thưa.

In [3]:
best = {1: 3, 2: 4}
print(best.get(1, 0))   # 3
print(best.get(0, 0))   # 0: chưa có khóa 0

3
0


# 3. Cách 1 — Brute Force: thử mọi dãy con

Có `2^n` dãy con (mỗi phần tử chọn hoặc không). Ta dùng **bitmask**: bit thứ `i` của số `mask` bằng `1` nghĩa là chọn `nums[i]`.

In [4]:
from typing import List


def max_len_bruteforce(nums: List[int], k: int) -> int:
    n = len(nums)
    best = 1
    for mask in range(1, 1 << n):
        sub = [nums[i] for i in range(n) if mask >> i & 1]
        if all((sub[j] + sub[j + 1]) % k == 0 for j in range(len(sub) - 1)):
            best = max(best, len(sub))
    return best


print(max_len_bruteforce([1, 2, 3, 4, 5], 3))    # 4
print(max_len_bruteforce([5, 10, 15], 5))        # 3
print(max_len_bruteforce([1, 3, 5, 7, 9], 4))    # 5

4
3
5


`2^n` với `n = 10^5` là không tưởng. Nhưng brute force rất hữu ích để kiểm tra chéo các cách nhanh hơn trên input nhỏ.

---

# 4. Cách 2 — DP "kết thúc tại `i`": O(n²)

Ý tưởng giống bài **Longest Increasing Subsequence (LIS)**:

> `f[i]` = độ dài dãy con hợp lệ dài nhất **kết thúc tại phần tử `nums[i]`**.

Một dãy hợp lệ kết thúc tại `i` hoặc chỉ có mỗi `nums[i]` (độ dài 1), hoặc là một dãy hợp lệ kết thúc tại `j < i` **nối thêm** `nums[i]`, với điều kiện `(nums[j] + nums[i]) % k == 0`:

```text
f[i] = 1 + max( f[j]  với j < i và (nums[j] + nums[i]) % k == 0 ),   mặc định 1
answer = max(f)
```

## Tính tay với ví dụ 1 (`k = 3`)

| i | nums[i] | các j hợp lệ (`nums[j] + nums[i]` chia hết cho 3) | f[i] |
|---:|---:|---|---:|
| 0 | 1 | — | 1 |
| 1 | 2 | j = 0 (1 + 2 = 3) | 1 + f[0] = 2 |
| 2 | 3 | — (3 + 1, 3 + 2 không chia hết) | 1 |
| 3 | 4 | j = 1 (2 + 4 = 6) | 1 + f[1] = 3 |
| 4 | 5 | j = 0 (1 + 5), j = 3 (4 + 5) | 1 + max(f[0], f[3]) = 4 |

`answer = 4`.

In [5]:
def max_len_quadratic(nums: List[int], k: int) -> int:
    n = len(nums)
    f = [1] * n
    for i in range(n):
        for j in range(i):
            if (nums[j] + nums[i]) % k == 0:
                f[i] = max(f[i], f[j] + 1)
    return max(f)


print(max_len_quadratic([1, 2, 3, 4, 5], 3))    # 4
print(max_len_quadratic([5, 10, 15], 5))        # 3
print(max_len_quadratic([1, 3, 5, 7, 9], 4))    # 5

4
3
5


`O(n²)` với `n = 10^5` là `10^10` phép toán — quá chậm.

---

# 5. Bước đột phá: chỉ số dư mới quan trọng

Nhìn lại vòng lặp trong của cách 2: ta duyệt **mọi** `j < i` và hỏi `(nums[j] + nums[i]) % k == 0`. Theo mục 2.3, điều kiện này tương đương:

```text
nums[j] % k == need,   với need = (k - nums[i] % k) % k
```

Vậy ta **không cần biết từng `j`**, chỉ cần biết:

> Trong các phần tử đã đi qua có số dư bằng `need`, phần tử nào có `f` lớn nhất?

Duy trì một bảng:

```text
best[r] = max f[j]  trên mọi j đã duyệt có nums[j] % k == r
```

Khi đó:

```text
r = nums[i] % k
need = (k - r) % k
f[i] = best.get(need, 0) + 1
best[r] = max(best[r], f[i])
```

Mỗi phần tử chỉ tốn `O(1)` → `O(n)` tổng cộng.

**Thứ tự quan trọng:** đọc `best[need]` **trước**, cập nhật `best[r]` **sau**. Nếu `r == need` (ví dụ `r = 0`), cập nhật trước sẽ khiến phần tử "ghép với chính nó".

## Tính tay với ví dụ 1 (`k = 3`)

| i | nums[i] | r | need | best[need] trước | f[i] | best sau |
|---:|---:|---:|---:|---:|---:|---|
| 0 | 1 | 1 | 2 | 0 | 1 | `{1: 1}` |
| 1 | 2 | 2 | 1 | 1 | 2 | `{1: 1, 2: 2}` |
| 2 | 3 | 0 | 0 | 0 | 1 | `{1: 1, 2: 2, 0: 1}` |
| 3 | 4 | 1 | 2 | 2 | 3 | `{1: 3, 2: 2, 0: 1}` |
| 4 | 5 | 2 | 1 | 3 | 4 | `{1: 3, 2: 4, 0: 1}` |

`answer = max f = 4` ✔.

# 6. Cách 3 — DP theo số dư: O(n)

In [6]:
def max_len_by_remainder(nums: List[int], k: int) -> int:
    best = {}                          # best[r]: dài nhất kết thúc bằng số có số dư r
    answer = 1
    for x in nums:
        r = x % k
        need = (k - r) % k
        cur = best.get(need, 0) + 1    # đọc TRƯỚC
        if cur > best.get(r, 0):
            best[r] = cur              # cập nhật SAU
        answer = max(answer, cur)
    return answer


print(max_len_by_remainder([1, 2, 3, 4, 5], 3))    # 4
print(max_len_by_remainder([5, 10, 15], 5))        # 3
print(max_len_by_remainder([1, 3, 5, 7, 9], 4))    # 5

4
3
5


In [7]:
def trace_by_remainder(nums: List[int], k: int) -> int:
    best = {}
    answer = 1
    print(f"nums = {nums}, k = {k}")
    for i, x in enumerate(nums):
        r = x % k
        need = (k - r) % k
        before = best.get(need, 0)
        cur = before + 1
        if cur > best.get(r, 0):
            best[r] = cur
        answer = max(answer, cur)
        print(f"i = {i}: x = {x:>2}, r = {r}, need = {need}, best[need] = {before} -> f = {cur} | best = {best}")
    print("Đáp án:", answer)
    return answer


trace_by_remainder([1, 2, 3, 4, 5], 3);

nums = [1, 2, 3, 4, 5], k = 3
i = 0: x =  1, r = 1, need = 2, best[need] = 0 -> f = 1 | best = {1: 1}
i = 1: x =  2, r = 2, need = 1, best[need] = 1 -> f = 2 | best = {1: 1, 2: 2}
i = 2: x =  3, r = 0, need = 0, best[need] = 0 -> f = 1 | best = {1: 1, 2: 2, 0: 1}
i = 3: x =  4, r = 1, need = 2, best[need] = 2 -> f = 3 | best = {1: 3, 2: 2, 0: 1}
i = 4: x =  5, r = 2, need = 1, best[need] = 3 -> f = 4 | best = {1: 3, 2: 4, 0: 1}
Đáp án: 4


# 7. Bước đột phá 2: dãy hợp lệ trông như thế nào?

Giả sử dãy hợp lệ bắt đầu bằng một số có số dư `x`. Số kế tiếp phải có số dư `y = need(x)`. Số sau nữa phải có số dư `need(y) = x` (vì quan hệ bù là đối xứng). Cứ thế:

```text
số dư:  x, y, x, y, x, ...      với y = (k - x) % k
```

Có hai trường hợp:

**Trường hợp A — `x == y`** (xảy ra khi `x = 0`, hoặc `2x = k`): mọi phần tử trong dãy có **cùng số dư `x`**, và **bất kỳ** hai số nào có số dư `x` cũng ghép được. Vậy lấy **tất cả** các số có số dư `x` → độ dài = số lượng số có số dư `x`.

**Trường hợp B — `x != y`:** dãy phải **xen kẽ** giữa lớp số dư `x` và lớp số dư `y`. Lọc riêng các số thuộc hai lớp này (giữ thứ tự), ta được một dãy nhãn, ví dụ:

```text
x x y x y y y x
```

Dãy con xen kẽ dài nhất lấy được **đúng một phần tử từ mỗi khối** nhãn giống nhau liên tiếp:

```text
[x x] [y] [x] [y y y] [x]   -> 5 khối -> độ dài 5
```

(Hai phần tử cùng khối không thể đứng kề nhau trong dãy xen kẽ, và giữa hai khối kề nhau luôn đổi nhãn, nên lấy một phần tử mỗi khối là tối ưu.)

Mỗi số dư `r` thuộc đúng một cặp `{r, need(r)}`. Ta gom các số theo **khóa cặp** `min(r, need(r))`, rồi với mỗi nhóm tính độ dài theo trường hợp A hoặc B.

## Áp dụng cho ví dụ 1 (`k = 3`)

Số dư: `1, 2, 0, 1, 2`.

- Cặp `{1, 2}`: dãy nhãn `1, 2, 1, 2` → 4 khối → **4**.
- Cặp `{0}` (trường hợp A): có 1 số → **1**.

Đáp án `max(4, 1) = 4` ✔.

## Áp dụng cho ví dụ 3 (`k = 4`)

Số dư: `1, 3, 1, 3, 1` — cặp `{1, 3}`, dãy nhãn xen kẽ hoàn hảo → 5 khối → **5** ✔.

# 8. Cách 4 — Gom theo cặp số dư và đếm khối: O(n)

In [8]:
def max_len_runs(nums: List[int], k: int) -> int:
    groups = {}                                  # khóa cặp -> dãy số dư theo thứ tự
    for x in nums:
        r = x % k
        key = min(r, (k - r) % k)
        groups.setdefault(key, []).append(r)

    answer = 0
    for key, rems in groups.items():
        if (2 * key) % k == 0:                   # trường hợp A: x == need(x)
            length = len(rems)
        else:                                    # trường hợp B: đếm số khối
            length = 1
            for a, b in zip(rems, rems[1:]):
                if a != b:
                    length += 1
        answer = max(answer, length)
    return answer


print(max_len_runs([1, 2, 3, 4, 5], 3))    # 4
print(max_len_runs([5, 10, 15], 5))        # 3
print(max_len_runs([1, 3, 5, 7, 9], 4))    # 5

4
3
5


In [9]:
def trace_runs(nums: List[int], k: int) -> int:
    groups = {}
    for x in nums:
        r = x % k
        key = min(r, (k - r) % k)
        groups.setdefault(key, []).append(r)
    answer = 0
    print(f"nums = {nums}, k = {k}, số dư = {[x % k for x in nums]}")
    for key, rems in groups.items():
        partner = (k - key) % k
        if (2 * key) % k == 0:
            length = len(rems)
            print(f"  cặp {{{key}}} (tự ghép): {rems} -> lấy hết {length} số")
        else:
            length = 1 + sum(1 for a, b in zip(rems, rems[1:]) if a != b)
            print(f"  cặp {{{key}, {partner}}}: dãy nhãn {rems} -> {length} khối")
        answer = max(answer, length)
    print("Đáp án:", answer)
    return answer


trace_runs([1, 2, 3, 4, 5], 3)
trace_runs([2, 7, 4, 5, 8, 1, 6, 9], 3);

nums = [1, 2, 3, 4, 5], k = 3, số dư = [1, 2, 0, 1, 2]
  cặp {1, 2}: dãy nhãn [1, 2, 1, 2] -> 4 khối
  cặp {0} (tự ghép): [0] -> lấy hết 1 số
Đáp án: 4
nums = [2, 7, 4, 5, 8, 1, 6, 9], k = 3, số dư = [2, 1, 1, 2, 2, 1, 0, 0]
  cặp {1, 2}: dãy nhãn [2, 1, 1, 2, 2, 1] -> 4 khối
  cặp {0} (tự ghép): [0, 0] -> lấy hết 2 số
Đáp án: 4


# 9. Phiên bản đúng format LeetCode

Ta nộp cách 3 (DP theo số dư): ngắn, một vòng lặp, dễ chứng minh.

In [10]:
from typing import List


class Solution:
    def maximumLength(self, nums: List[int], k: int) -> int:
        best = {}
        answer = 1
        for x in nums:
            r = x % k
            need = (k - r) % k
            cur = best.get(need, 0) + 1
            if cur > best.get(r, 0):
                best[r] = cur
            answer = max(answer, cur)
        return answer

In [11]:
sol = Solution()

tests = [
    ([1, 2, 3, 4, 5], 3, 4, "ví dụ 1"),
    ([5, 10, 15], 5, 3, "ví dụ 2"),
    ([1, 3, 5, 7, 9], 4, 5, "ví dụ 3"),
    ([1, 1], 5, 1, "không cặp nào ghép được -> 1"),
    ([7, 8, 9], 1, 3, "k = 1: mọi tổng đều chia hết"),
    ([1, 3, 2, 5, 4], 2, 3, "k = 2: cùng tính chẵn lẻ"),
    ([2, 2, 2, 2], 4, 4, "2x = k: tự ghép"),
    ([10**9, 10**9], 10**5, 2, "số lớn"),
]

for nums, k, expected, name in tests:
    result = sol.maximumLength(nums, k)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:32} -> {result}")

PASS | ví dụ 1                          -> 4
PASS | ví dụ 2                          -> 3
PASS | ví dụ 3                          -> 5
PASS | không cặp nào ghép được -> 1     -> 1
PASS | k = 1: mọi tổng đều chia hết     -> 3
PASS | k = 2: cùng tính chẵn lẻ         -> 3
PASS | 2x = k: tự ghép                  -> 4
PASS | số lớn                           -> 2


Giải thích test `k = 2`: `a + b` chẵn khi `a, b` cùng tính chẵn lẻ. Lớp lẻ `{1, 3, 5}` dài 3, lớp chẵn `{2, 4}` dài 2 → đáp án 3.

Thử với mảng lớn:

In [12]:
import random
import time

random.seed(4110)
big = [random.randint(1, 10**9) for _ in range(10**5)]
t0 = time.perf_counter()
r3 = sol.maximumLength(big, 7)
r4 = max_len_runs(big, 7)
print(f"n = 10^5, k = 7: cách 3 = {r3}, cách 4 = {r4} ({time.perf_counter() - t0:.2f}s cho cả hai)")

n = 10^5, k = 7: cách 3 = 14377, cách 4 = 14377 (0.02s cho cả hai)


# 10. Vì sao cách 3 đúng?

**Bất biến:** sau khi duyệt xong các phần tử `nums[0..i-1]`, với mọi số dư `r`:

```text
best[r] = độ dài dãy con hợp lệ dài nhất (trong nums[0..i-1]) kết thúc bằng một số có số dư r
```

(nếu chưa có, coi là `0`).

Khi xét `nums[i]` với số dư `r`:

- Một dãy hợp lệ kết thúc tại `nums[i]` có dạng: `[nums[i]]` (độ dài 1), hoặc một dãy hợp lệ kết thúc tại `nums[j]` (`j < i`) nối thêm `nums[i]`, với `nums[j] % k == need`.
- Dài nhất trong các dãy đó là `best[need] + 1` (với `best[need] = 0` thì là chính `[nums[i]]`).
- Ta đọc `best[need]` **trước** khi cập nhật, nên nó chỉ gồm các `j < i` — không bao giờ ghép `nums[i]` với chính nó.
- Cập nhật `best[r] = max(best[r], f[i])` giữ bất biến cho bước sau.

Đáp án là `max` của mọi `f[i]`, vì dãy hợp lệ dài nhất phải kết thúc tại một phần tử nào đó.

---

## Trực giác

> Mỗi phần tử hỏi: *"Trong những bạn đã đi trước có số dư 'hợp cạ' với mình, chuỗi dài nhất là bao nhiêu?"* — rồi nối mình vào chuỗi đó. Ta không cần nhớ từng bạn, chỉ cần nhớ **kỷ lục theo từng số dư**.

# 11. Độ phức tạp

Gọi `n = len(nums)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ |
|---|---|---:|---:|
| 1. Brute Force | thử mọi dãy con | `O(2^n · n)` | `O(n)` |
| 2. DP kiểu LIS | `f[i]` từ mọi `j < i` | `O(n²)` | `O(n)` |
| 3. DP theo số dư | `best[r]`, đọc `best[need]` | `O(n)` | `O(min(n, k))` |
| 4. Đếm khối | gom theo cặp số dư | `O(n)` | `O(n)` |

Bộ nhớ của cách 3 là số số dư khác nhau thực sự xuất hiện — không quá `n` và không quá `k`.

# 12. Đối chiếu bốn cách trên dữ liệu ngẫu nhiên

In [13]:
random.seed(2026)
for _ in range(3000):
    n = random.randint(2, 10)
    k = random.randint(1, 7)
    nums = [random.randint(1, 20) for _ in range(n)]
    expected = max_len_bruteforce(nums, k)
    assert max_len_quadratic(nums, k) == expected, (nums, k)
    assert max_len_by_remainder(nums, k) == expected, (nums, k)
    assert max_len_runs(nums, k) == expected, (nums, k)
    assert sol.maximumLength(nums, k) == expected, (nums, k)

print("3000 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!")

3000 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!


# 13. Những lỗi dễ mắc

## Lỗi 1 — Tính số dư bù là `k - r`, quên `% k`

Với `r = 0`, `need = k` — không số dư nào bằng `k`, nên các bội của `k` không bao giờ ghép được với nhau.

In [14]:
def wrong_need_without_mod(nums, k):
    best = {}
    answer = 1
    for x in nums:
        r = x % k
        need = k - r                        # SAI: thiếu % k
        cur = best.get(need, 0) + 1
        best[r] = max(best.get(r, 0), cur)
        answer = max(answer, cur)
    return answer


print("Sai :", wrong_need_without_mod([5, 10, 15], 5))
print("Đúng:", sol.maximumLength([5, 10, 15], 5))

Sai : 1
Đúng: 3


## Lỗi 2 — Hiểu nhầm thành subarray (đoạn liên tiếp)

Nếu chỉ xét các đoạn liên tiếp, ví dụ 1 chỉ đạt `2` (`[1, 2]` hoặc `[4, 5]`), vì số `3` chặn ở giữa.

In [15]:
def wrong_as_subarray(nums, k):
    best = cur = 1
    for i in range(1, len(nums)):
        cur = cur + 1 if (nums[i - 1] + nums[i]) % k == 0 else 1
        best = max(best, cur)
    return best


print("Sai :", wrong_as_subarray([1, 2, 3, 4, 5], 3))
print("Đúng:", sol.maximumLength([1, 2, 3, 4, 5], 3))

Sai : 2
Đúng: 4


## Lỗi 3 — Ở cách đếm khối, quên trường hợp "tự ghép"

Khi `x == need(x)` (ví dụ `x = 0`), mọi phần tử cùng nhãn — dãy nhãn chỉ có **1 khối** — nhưng thật ra lấy được **tất cả**.

In [16]:
def wrong_runs_no_self_pair(nums, k):
    groups = {}
    for x in nums:
        r = x % k
        groups.setdefault(min(r, (k - r) % k), []).append(r)
    answer = 0
    for rems in groups.values():
        length = 1 + sum(1 for a, b in zip(rems, rems[1:]) if a != b)   # SAI với nhóm tự ghép
        answer = max(answer, length)
    return answer


print("Sai :", wrong_runs_no_self_pair([5, 10, 15], 5))
print("Đúng:", sol.maximumLength([5, 10, 15], 5))

Sai : 1
Đúng: 3


## Lỗi 4 — Khởi tạo đáp án bằng `0`

Dãy một phần tử luôn hợp lệ, nên đáp án ít nhất là `1`. Nếu chỉ cập nhật đáp án khi "ghép được một cặp", input như `[1, 1]` với `k = 5` sẽ trả về `0`. (Code ở cách 3 an toàn vì luôn cập nhật `answer` bằng `cur >= 1`.)

---

## Lỗi 5 — Cập nhật `best[r]` trước khi đọc `best[need]`

Khi `r == need` (ví dụ `r = 0`), phần tử sẽ ghép với **chính nó**: một mảng chỉ có một bội của `k` sẽ cho đáp án `2`.

In [17]:
def wrong_update_first(nums, k):
    best = {}
    answer = 1
    for x in nums:
        r = x % k
        need = (k - r) % k
        best[r] = max(best.get(r, 0), 1)       # SAI: ghi phần tử hiện tại vào bảng TRƯỚC...
        cur = best.get(need, 0) + 1            # ...nên khi r == need, nó tự ghép với chính nó
        best[r] = max(best[r], cur)
        answer = max(answer, cur)
    return answer


print("Sai :", wrong_update_first([3, 1], 3))     # chỉ có một bội của 3 -> đáp án đúng là 1
print("Đúng:", sol.maximumLength([3, 1], 3))

Sai : 2
Đúng: 1


## Lỗi 6 — Dùng DP `O(n²)` cho `n = 10^5`

Đúng nhưng TLE. Dấu hiệu tối ưu: điều kiện ghép chỉ phụ thuộc vào một "loại" nhỏ của phần tử (ở đây là số dư) → gom trạng thái theo loại.

# 14. Mẹo nhận diện

1. Thấy **"tổng / hiệu chia hết cho `k`"** → chuyển sang làm việc với **số dư** `x % k`.
2. Thấy **"dãy con dài nhất"** với điều kiện chỉ giữa **hai phần tử kề nhau** → DP **"kết thúc tại `i`"** kiểu LIS.
3. Nếu điều kiện kề nhau chỉ phụ thuộc vào một **thuộc tính có ít giá trị** (số dư, ký tự, giá trị...) → thay vòng lặp `j` bằng bảng **`best[thuộc tính]`** → từ `O(n²)` xuống `O(n)`.
4. Nếu dãy hợp lệ buộc phải **xen kẽ hai loại** → đáp án là **số khối** trong dãy đã lọc.

Các bài cùng họ:

- *Find the Maximum Length of Valid Subsequence I / II* (LeetCode 3201 / 3202) — các tổng kề nhau phải có **cùng số dư** khi chia cho `2` / `k`.
- *Longest Arithmetic Subsequence of Given Difference* (LeetCode 1218) — `dp[x] = dp[x - diff] + 1`, cùng kiểu "DP theo giá trị".
- *Longest Increasing Subsequence* (LeetCode 300) — DP "kết thúc tại `i`" gốc.

# 15. Biến thể

## Biến thể 1 — Tổng kề nhau phải có số dư bằng `c` (thay vì `0`)

Chỉ cần đổi công thức số dư bù: `need = (c - r) % k`. Python luôn trả về số dư không âm với `k > 0`, kể cả khi `c - r` âm.

In [18]:
print((-1) % 5, (2 - 4) % 5)        # 4 3: Python luôn cho số dư trong 0..k-1


def max_len_sum_mod_c(nums: List[int], k: int, c: int) -> int:
    best = {}
    answer = 1
    for x in nums:
        r = x % k
        need = (c - r) % k
        cur = best.get(need, 0) + 1
        if cur > best.get(r, 0):
            best[r] = cur
        answer = max(answer, cur)
    return answer


print(max_len_sum_mod_c([1, 2, 3, 4, 5], 3, 0))   # giống bài gốc: 4
print(max_len_sum_mod_c([1, 2, 3, 4, 5], 3, 1))   # các tổng kề nhau chia 3 dư 1

4 3
4
3


## Biến thể 2 — **Hiệu** kề nhau chia hết cho `k`

`(a - b) % k == 0` ⇔ `a % k == b % k`: mọi phần tử trong dãy phải **cùng số dư**. Đáp án = số lượng lớn nhất của một lớp số dư.

In [19]:
from collections import Counter


def max_len_diff_divisible(nums: List[int], k: int) -> int:
    return max(Counter(x % k for x in nums).values())


print(max_len_diff_divisible([1, 4, 7, 2, 10], 3))   # số dư 1: [1, 4, 7, 10] -> 4

4


# 16. Template tư duy

```python
# Dãy con dài nhất, điều kiện giữa hai phần tử kề nhau chỉ phụ thuộc "loại" của chúng
best = {}                         # best[loại] = độ dài dài nhất kết thúc bằng loại đó
answer = 1
for x in nums:
    t = type_of(x)                # ví dụ: x % k
    cur = 1 + max(best.get(u, 0) for u in compatible_types(t))   # thường chỉ 1 loại
    best[t] = max(best.get(t, 0), cur)     # cập nhật SAU khi đọc
    answer = max(answer, cur)
```

# 17. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lập bảng `i | x | r | need | best[need] | f | best` cho:

```python
nums = [2, 7, 4, 5, 8, 1], k = 3
```

Đáp án là bao nhiêu? Chỉ ra một dãy con hợp lệ đạt độ dài đó. Kiểm tra bằng cell dưới.

In [20]:
# Kiểm tra Bài 1
trace_by_remainder([2, 7, 4, 5, 8, 1], 3);

nums = [2, 7, 4, 5, 8, 1], k = 3
i = 0: x =  2, r = 2, need = 1, best[need] = 0 -> f = 1 | best = {2: 1}
i = 1: x =  7, r = 1, need = 2, best[need] = 1 -> f = 2 | best = {2: 1, 1: 2}
i = 2: x =  4, r = 1, need = 2, best[need] = 1 -> f = 2 | best = {2: 1, 1: 2}
i = 3: x =  5, r = 2, need = 1, best[need] = 2 -> f = 3 | best = {2: 3, 1: 2}
i = 4: x =  8, r = 2, need = 1, best[need] = 2 -> f = 3 | best = {2: 3, 1: 2}
i = 5: x =  1, r = 1, need = 2, best[need] = 3 -> f = 4 | best = {2: 3, 1: 4}
Đáp án: 4


## Bài 2 — Dự đoán output

```python
k = 6
print([(k - r) % k for r in range(k)])
print([min(r, (k - r) % k) for r in range(k)])
print([(2 * r) % k == 0 for r in range(k)])
```

In [21]:
k = 6
print([(k - r) % k for r in range(k)])
print([min(r, (k - r) % k) for r in range(k)])
print([(2 * r) % k == 0 for r in range(k)])

[0, 5, 4, 3, 2, 1]
[0, 1, 2, 3, 2, 1]
[True, False, False, True, False, False]


## Bài 3 — Hoàn thiện DP theo số dư

In [22]:
def exercise_3_by_remainder(nums: List[int], k: int) -> int:
    best = {}
    answer = 1
    for x in nums:
        # TODO 1: tính r và need
        # TODO 2: cur = best[need] + 1 (dùng get)
        # TODO 3: cập nhật best[r] và answer
        pass
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_by_remainder([1, 2, 3, 4, 5], 3) == 4
# assert exercise_3_by_remainder([5, 10, 15], 5) == 3

## Bài 4 — Tự viết cách đếm khối

Không nhìn code phía trên. Nhớ xử lý trường hợp "tự ghép".

In [23]:
def exercise_4_runs(nums: List[int], k: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_runs([1, 2, 3, 4, 5], 3) == 4
# assert exercise_4_runs([5, 10, 15], 5) == 3
# assert exercise_4_runs([2, 2, 2, 2], 4) == 4

## Bài 5 — Biến thể 3201: các tổng kề nhau có **cùng tính chẵn lẻ**

Dãy con hợp lệ nếu `(sub[0] + sub[1]) % 2 == (sub[1] + sub[2]) % 2 == ...` (tất cả các tổng kề nhau cùng chẵn, hoặc cùng lẻ). Tìm độ dài lớn nhất.

```python
max_len_same_parity_sum([1, 2, 3, 4])        -> 4    # tổng kề nhau đều lẻ
max_len_same_parity_sum([1, 2, 1, 1, 2, 1, 2]) -> 6
max_len_same_parity_sum([1, 3])              -> 2
```

Gợi ý: "tất cả tổng chẵn" hoặc "tất cả tổng lẻ" — mỗi trường hợp chính là bài gốc với `k = 2` và một giá trị `c` (biến thể 1).

In [24]:
def exercise_5_max_len_same_parity_sum(nums: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_max_len_same_parity_sum([1, 2, 3, 4]) == 4
# assert exercise_5_max_len_same_parity_sum([1, 2, 1, 1, 2, 1, 2]) == 6
# assert exercise_5_max_len_same_parity_sum([1, 3]) == 2

## Bài 6 — Mini challenge: chỉ ra dãy con

Trả về **một dãy con hợp lệ dài nhất** (danh sách các giá trị, theo thứ tự trong `nums`).

```python
longest_valid_subsequence([1, 2, 3, 4, 5], 3)   -> độ dài 4, ví dụ [1, 2, 4, 5]
longest_valid_subsequence([5, 10, 15], 5)       -> [5, 10, 15]
```

Gợi ý: ngoài `best[r]`, lưu thêm `best_idx[r]` (chỉ số phần tử đạt kỷ lục) và mảng `prev[i]` (phần tử đứng trước `i` trong dãy tốt nhất kết thúc tại `i`).

In [25]:
def exercise_6_longest_valid_subsequence(nums: List[int], k: int) -> List[int]:
    # TODO
    raise NotImplementedError


# res = exercise_6_longest_valid_subsequence([1, 2, 3, 4, 5], 3)
# assert len(res) == 4 and all((a + b) % 3 == 0 for a, b in zip(res, res[1:]))

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `r = x % k`, `need = (k - r) % k`.
- Mức 2: `cur = best.get(need, 0) + 1`.
- Mức 3: `best[r] = max(best.get(r, 0), cur)`, `answer = max(answer, cur)`; cuối cùng `return answer`.

**Bài 4**

- Mức 1: khóa nhóm `min(r, (k - r) % k)`, giữ thứ tự các số dư trong nhóm.
- Mức 2: nhóm tự ghép khi `(2 * key) % k == 0` → độ dài = số phần tử.
- Mức 3: nhóm còn lại → `1 + số chỗ đổi nhãn`.

**Bài 5**

- Mức 1: "mọi tổng kề nhau chẵn" ⇔ mọi phần tử cùng tính chẵn lẻ; "mọi tổng kề nhau lẻ" ⇔ chẵn/lẻ xen kẽ.
- Mức 2: dùng `max_len_sum_mod_c(nums, 2, 0)` và `max_len_sum_mod_c(nums, 2, 1)`.
- Mức 3: đáp án là `max` của hai kết quả.

**Bài 6**

- Mức 1: khi tính `cur` cho phần tử `i`, phần tử đứng trước nó (nếu có) là `best_idx[need]`.
- Mức 2: lưu `prev[i] = best_idx.get(need, -1)`; khi cập nhật kỷ lục `best[r]`, đồng thời `best_idx[r] = i`.
- Mức 3: lưu chỉ số `end` đạt `answer`, rồi lần ngược `end -> prev[end] -> ...` và đảo ngược danh sách.

# 18. Đáp án tham khảo

Bạn nên tự làm trước khi mở phần này.

## Đáp án Bài 1

`nums = [2, 7, 4, 5, 8, 1]`, `k = 3` → số dư `2, 1, 1, 2, 2, 1`.

| i | x | r | need | best[need] | f | best sau |
|---:|---:|---:|---:|---:|---:|---|
| 0 | 2 | 2 | 1 | 0 | 1 | `{2: 1}` |
| 1 | 7 | 1 | 2 | 1 | 2 | `{2: 1, 1: 2}` |
| 2 | 4 | 1 | 2 | 1 | 2 | `{2: 1, 1: 2}` |
| 3 | 5 | 2 | 1 | 2 | 3 | `{2: 3, 1: 2}` |
| 4 | 8 | 2 | 1 | 2 | 3 | `{2: 3, 1: 2}` |
| 5 | 1 | 1 | 2 | 3 | 4 | `{2: 3, 1: 4}` |

Đáp án `4`, ví dụ dãy `[2, 7, 5, 1]` (tổng kề nhau `9, 12, 6`). Theo cách đếm khối: dãy nhãn `2 | 1 1 | 2 2 | 1` có 4 khối.

## Đáp án Bài 2

```text
[0, 5, 4, 3, 2, 1]
[0, 1, 2, 3, 2, 1]
[True, False, False, True, False, False]
```

Với `k = 6`: số dư `0` và `3` tự ghép; các cặp còn lại là `{1, 5}`, `{2, 4}`.

In [26]:
# Đáp án Bài 3
def answer_3_by_remainder(nums: List[int], k: int) -> int:
    best = {}
    answer = 1
    for x in nums:
        r = x % k
        need = (k - r) % k
        cur = best.get(need, 0) + 1
        best[r] = max(best.get(r, 0), cur)
        answer = max(answer, cur)
    return answer


assert answer_3_by_remainder([1, 2, 3, 4, 5], 3) == 4
assert answer_3_by_remainder([5, 10, 15], 5) == 3
assert answer_3_by_remainder([2, 7, 4, 5, 8, 1], 3) == 4
print("Bài 3: PASS")

Bài 3: PASS


In [27]:
# Đáp án Bài 4
def answer_4_runs(nums: List[int], k: int) -> int:
    groups = {}
    for x in nums:
        r = x % k
        groups.setdefault(min(r, (k - r) % k), []).append(r)
    answer = 0
    for key, rems in groups.items():
        if (2 * key) % k == 0:
            length = len(rems)
        else:
            length = 1 + sum(1 for a, b in zip(rems, rems[1:]) if a != b)
        answer = max(answer, length)
    return answer


assert answer_4_runs([1, 2, 3, 4, 5], 3) == 4
assert answer_4_runs([5, 10, 15], 5) == 3
assert answer_4_runs([2, 2, 2, 2], 4) == 4
print("Bài 4: PASS")

Bài 4: PASS


In [28]:
# Đáp án Bài 5
def answer_5_max_len_same_parity_sum(nums: List[int]) -> int:
    return max(max_len_sum_mod_c(nums, 2, 0), max_len_sum_mod_c(nums, 2, 1))


def brute_same_parity_sum(nums):
    n = len(nums)
    best = 1
    for mask in range(1, 1 << n):
        sub = [nums[i] for i in range(n) if mask >> i & 1]
        sums = {(a + b) % 2 for a, b in zip(sub, sub[1:])}
        if len(sums) <= 1:
            best = max(best, len(sub))
    return best


assert answer_5_max_len_same_parity_sum([1, 2, 3, 4]) == 4
assert answer_5_max_len_same_parity_sum([1, 2, 1, 1, 2, 1, 2]) == 6
assert answer_5_max_len_same_parity_sum([1, 3]) == 2
random.seed(5)
for _ in range(500):
    arr = [random.randint(1, 9) for _ in range(random.randint(2, 9))]
    assert answer_5_max_len_same_parity_sum(arr) == brute_same_parity_sum(arr), arr
print("Bài 5: PASS (kể cả 500 test ngẫu nhiên)")

Bài 5: PASS (kể cả 500 test ngẫu nhiên)


In [29]:
# Đáp án Bài 6
def answer_6_longest_valid_subsequence(nums: List[int], k: int) -> List[int]:
    best, best_idx = {}, {}
    prev = [-1] * len(nums)
    answer, end = 0, -1
    for i, x in enumerate(nums):
        r = x % k
        need = (k - r) % k
        cur = best.get(need, 0) + 1
        prev[i] = best_idx[need] if need in best_idx else -1
        if cur > best.get(r, 0):
            best[r] = cur
            best_idx[r] = i
        if cur > answer:
            answer, end = cur, i
    result = []
    while end != -1:
        result.append(nums[end])
        end = prev[end]
    return result[::-1]


res = answer_6_longest_valid_subsequence([1, 2, 3, 4, 5], 3)
print(res)
assert len(res) == 4 and all((a + b) % 3 == 0 for a, b in zip(res, res[1:]))
assert answer_6_longest_valid_subsequence([5, 10, 15], 5) == [5, 10, 15]
random.seed(6)
for _ in range(500):
    arr = [random.randint(1, 20) for _ in range(random.randint(2, 10))]
    kk = random.randint(1, 6)
    seq = answer_6_longest_valid_subsequence(arr, kk)
    assert len(seq) == max_len_bruteforce(arr, kk)
    assert all((a + b) % kk == 0 for a, b in zip(seq, seq[1:]))
    it = iter(arr)
    assert all(v in it for v in seq)                 # đúng là dãy con của arr
print("Bài 6: PASS (độ dài tối ưu, hợp lệ, đúng là dãy con)")

[1, 2, 4, 5]


Bài 6: PASS (độ dài tối ưu, hợp lệ, đúng là dãy con)


# 19. Tổng kết

Bài **Find the Maximum Length of Valid Subsequence III** dạy hai bước tối ưu kinh điển:

1. **Chuyển sang số dư:** `(a + b) % k == 0` ⇔ `b % k == (k - a % k) % k`.
2. **DP kết thúc tại `i`** → **gom theo số dư**: `best[r]` thay cho việc duyệt mọi `j`.

Và một góc nhìn cấu trúc: dãy hợp lệ **xen kẽ hai lớp số dư** (hoặc toàn một lớp tự ghép) → đếm khối.

## Công thức nhớ nhanh

```text
r    = x % k
need = (k - r) % k
cur  = best[need] + 1          (đọc trước)
best[r] = max(best[r], cur)    (cập nhật sau)
```

## Phiên bản nên nhớ

```python
class Solution:
    def maximumLength(self, nums: List[int], k: int) -> int:
        best = {}
        answer = 1
        for x in nums:
            r = x % k
            need = (k - r) % k
            cur = best.get(need, 0) + 1
            if cur > best.get(r, 0):
                best[r] = cur
            answer = max(answer, cur)
        return answer
```

## Độ phức tạp

```text
Time:  O(n)
Space: O(min(n, k))
```

# 20. Checklist trước khi submit

- [ ] Đề nói **subsequence** (được bỏ phần tử), không phải subarray?
- [ ] Số dư bù là `(k - r) % k` (có `% k` để xử lý `r = 0`)?
- [ ] Đọc `best[need]` **trước**, cập nhật `best[r]` **sau** (tránh tự ghép với chính mình)?
- [ ] Đáp án khởi tạo `>= 1` (một phần tử luôn hợp lệ)?
- [ ] Đã test `k = 1`, `k = 2`, trường hợp `2r = k`, mảng không có cặp nào ghép được?
- [ ] Không dùng DP `O(n²)` cho `n = 10^5`?